In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Helper to produce compact meta and top cmdb_id counts (top 50)
def meta_and_cmdb_top(df):
    rows = df.shape[0]
    if rows > 0 and 'timestamp' in df.columns and df['timestamp'].notna().any():
        min_ts = pd.to_datetime(df['timestamp'].min(), unit='s', utc=True).tz_convert(tz)
        max_ts = pd.to_datetime(df['timestamp'].max(), unit='s', utc=True).tz_convert(tz)
        min_ts_str = min_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
        max_ts_str = max_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        min_ts_str = None
        max_ts_str = None
    if 'cmdb_id' in df.columns:
        cmdb_counts = df['cmdb_id'].value_counts()
        distinct_cmdb_count = int(cmdb_counts.shape[0])
        cmdb_top = cmdb_counts.reset_index().rename(columns={'index':'cmdb_id', 'cmdb_id':'count'}).head(50)
    else:
        distinct_cmdb_count = 0
        cmdb_top = pd.DataFrame(columns=['cmdb_id','count'])
    meta = pd.DataFrame([{
        'rows': int(rows),
        'min_timestamp_UTC+8': min_ts_str,
        'max_timestamp_UTC+8': max_ts_str,
        'distinct_cmdb_count': distinct_cmdb_count
    }])
    return meta, cmdb_top

# Summaries for metric.csv
metric_meta, metric_cmdb_top = meta_and_cmdb_top(metric)
if 'kpi_name' in metric.columns:
    metric_kpi_counts = metric['kpi_name'].value_counts().reset_index().rename(columns={'index':'kpi_name', 'kpi_name':'count'}).head(50)
else:
    metric_kpi_counts = pd.DataFrame(columns=['kpi_name','count'])

# Summaries for trace.csv
trace_meta, trace_cmdb_top = meta_and_cmdb_top(trace)
if 'trace_name' in trace.columns:
    trace_trace_counts = trace['trace_name'].value_counts().reset_index().rename(columns={'index':'trace_name', 'trace_name':'count'}).head(50)
else:
    trace_trace_counts = pd.DataFrame(columns=['trace_name','count'])

# Summaries for log.csv
log_meta, log_cmdb_top = meta_and_cmdb_top(log)
if 'log_name' in log.columns:
    log_log_counts = log['log_name'].value_counts().reset_index().rename(columns={'index':'log_name', 'log_name':'count'}).head(50)
else:
    log_log_counts = pd.DataFrame(columns=['log_name','count'])

# Summaries for error_logs.csv (including top 5 rows of timestamp, cmdb_id, message)
error_meta, error_cmdb_top = meta_and_cmdb_top(error_logs)
if {'timestamp','cmdb_id','message'}.issubset(error_logs.columns) and error_logs.shape[0] > 0:
    error_top5 = error_logs[['timestamp','cmdb_id','message']].copy().head(5)
    # Convert timestamp column properly
    error_top5['timestamp'] = pd.to_datetime(error_top5['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_top5 = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Return compact summaries (JSON-friendly)
metric_meta, metric_cmdb_top, metric_kpi_counts, \
trace_meta, trace_cmdb_top, trace_trace_counts, \
log_meta, log_cmdb_top, log_log_counts, \
error_meta, error_cmdb_top, error_top5

```
Out[1]:
```
```python
# Summarize the previously computed summaries in plain English as a compact string.
# This cell reuses variables computed earlier: metric_meta, metric_cmdb_top, metric_kpi_counts,
# trace_meta, trace_cmdb_top, trace_trace_counts, log_meta, log_cmdb_top, log_log_counts,
# error_meta, error_cmdb_top, error_top5.

def top_list_to_str(df, key_col, val_col='count', top_n=10):
    if df is None or df.shape[0] == 0:
        return "None"
    items = []
    for _, r in df.head(top_n).iterrows():
        items.append(f"{r[key_col]}({int(r[val_col])})")
    more = ""
    if df.shape[0] > top_n:
        more = f", ... (+{df.shape[0]-top_n} more)"
    return ", ".join(items) + more

m_meta = metric_meta.iloc[0]
t_meta = trace_meta.iloc[0]
l_meta = log_meta.iloc[0]
e_meta = error_meta.iloc[0]

summary = (
    "Summary of telemetry CSV files (timestamps shown in UTC+8):\n\n"
    "1) metric.csv:\n"
    f"   - rows: {int(m_meta['rows'])}\n"
    f"   - time range: {m_meta['min_timestamp_UTC+8']} to {m_meta['max_timestamp_UTC+8']}\n"
    f"   - distinct cmdb_ids: {int(m_meta['distinct_cmdb_count'])}\n"
    f"   - top cmdb_ids (top 10): {top_list_to_str(metric_cmdb_top, 'cmdb_id')}\n"
    f"   - top kpi_names (top 10): {top_list_to_str(metric_kpi_counts, 'kpi_name')}\n\n"
    "2) trace.csv:\n"
    f"   - rows: {int(t_meta['rows'])}\n"
    f"   - time range: {t_meta['min_timestamp_UTC+8']} to {t_meta['max_timestamp_UTC+8']}\n"
    f"   - distinct cmdb_ids: {int(t_meta['distinct_cmdb_count'])}\n"
    f"   - top cmdb_ids (top 10): {top_list_to_str(trace_cmdb_top, 'cmdb_id')}\n"
    f"   - top trace_names (top 10): {top_list_to_str(trace_trace_counts, 'trace_name')}\n\n"
    "3) log.csv:\n"
    f"   - rows: {int(l_meta['rows'])}\n"
    f"   - time range: {l_meta['min_timestamp_UTC+8']} to {l_meta['max_timestamp_UTC+8']}\n"
    f"   - distinct cmdb_ids: {int(l_meta['distinct_cmdb_count'])}\n"
    f"   - top cmdb_ids (top 10): {top_list_to_str(log_cmdb_top, 'cmdb_id')}\n"
    f"   - top log_names (top 10): {top_list_to_str(log_log_counts, 'log_name')}\n\n"
    "4) error_logs.csv:\n"
    f"   - rows: {int(e_meta['rows'])}\n"
    f"   - time range: {e_meta['min_timestamp_UTC+8']} to {e_meta['max_timestamp_UTC+8']}\n"
    f"   - distinct cmdb_ids: {int(e_meta['distinct_cmdb_count'])}\n"
    "   - top 5 error rows (timestamp, cmdb_id, message preview):\n"
)

# append top5 error rows with truncated message preview
if error_top5.shape[0] > 0:
    for _, r in error_top5.iterrows():
        msg = str(r['message'])
        preview = (msg[:140] + "...") if len(msg) > 140 else msg
        summary += f"     - {r['timestamp']}, {r['cmdb_id']}, {preview}\n"
else:
    summary += "     - None\n"

# Display the summary string (compact, plain English)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     rows      min_timestamp_UTC+8      max_timestamp_UTC+8  distinct_cmdb_count
0  149812  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                   59,             count  count
0       adservice  10230
1      adservice2   9990
2     frontend2-0   5280
3      frontend-0   5160
4      frontend-1   5160
..            ...    ...
45         node-1   1692
46         node-3   1692
47         node-5   1692
48  redis-cart2-0    360
49   redis-cart-0    360

[50 rows x 2 columns],                                                 count  count
0   container.node-5.container_network_transmit_er...   1050
1   container.node-5.container_network_transmit_pa...   1050
2   container.node-5.container_network_receive_err...   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_receive_MB....   1050
..                                                ...    ...
45  container.node-5.container_fs_sector_writes./d...    990
46  container.node-5.container_fs_sector_reads./de...    990
47  container.node-5.container_fs_reads_merged./de...    990
48  container.node-5.container_ulimits_soft.max_op...    990
49             container.node-5.container_threads_max    990

[50 rows x 2 columns],     rows      min_timestamp_UTC+8      max_timestamp_UTC+8  distinct_cmdb_count
0  35824  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                   40,                        count  count
0                 frontend-0   2756
1                 frontend-1   2684
2          checkoutservice-1   2656
3                 frontend-2   2620
4          checkoutservice-0   2556
..                       ...    ...
35        currencyservice2-0    236
36  productcatalogservice2-0    236
37              adservice2-0    120
38           emailservice2-0    116
39         paymentservice2-0    116

[40 rows x 2 columns],                                     count  count
0     trace.from_frontend-0.duration_mean    629
1      trace.from_frontend-0.duration_p95    629
2        trace.from_frontend-0.error_rate    629
3         trace.from_frontend-0.row_count    629
4     trace.from_frontend-1.duration_mean    611
..                                    ...    ...
45    trace.from_frontend2-0.duration_p95    209
46      trace.from_frontend2-0.error_rate    209
47       trace.from_frontend2-0.row_count    209
48   trace.to_currencyservice-0.row_count    180
49  trace.to_currencyservice-0.error_rate    180

[50 rows x 2 columns],    rows      min_timestamp_UTC+8      max_timestamp_UTC+8  distinct_cmdb_count
0  1812  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                   31,                       count  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
..                      ...    ...
26  recommendationservice-2     60
27        shippingservice-2     60
28        shippingservice-1     60
29         paymentservice-1     58
30             redis-cart-0     14

[31 rows x 2 columns],              count  count
0  log.error_count    906
1    log.row_count    906,    rows      min_timestamp_UTC+8      max_timestamp_UTC+8  distinct_cmdb_count
0  1488  2022-03-21 15:00:00 CST  2022-03-21 15:29:58 CST                    3,         count  count
0  frontend-0    748
1  frontend-1    445
2  frontend-2    295,                  timestamp     cmdb_id                                            message
0  2022-03-21 15:00:00 CST  frontend-1  severity: warning, message: failed to retrieve...
1  2022-03-21 15:00:02 CST  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21 15:00:03 CST  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21 15:00:05 CST  frontend-0  severity: warning, message: failed to retrieve...
4  2022-03-21 15:00:06 CST  frontend-2  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Load metric.csv (reuse if already loaded in kernel)
metric = pd.read_csv('metric.csv')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using all rows (no time filtering)
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# Merge thresholds back to metric (keeps full series)
metric_with_p95 = metric.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Convert timestamps to timezone-aware datetimes in UTC+8 for window filtering and earliest timestamp reporting
metric_with_p95['timestamp_dt'] = (
    pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True)
    .dt.tz_convert(tz)
)

# 2) Mark rows within the incident window and anomalies (value >= global P95)
start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))  # exclusive

metric_with_p95['is_in_window'] = (metric_with_p95['timestamp_dt'] >= start) & (metric_with_p95['timestamp_dt'] < end)
metric_with_p95['is_anomaly'] = metric_with_p95['is_in_window'] & (metric_with_p95['value'] >= metric_with_p95['p95'])

# 3) Aggregate anomaly counts per cmdb_id within the incident window (sum across KPIs)
anomalies_window = metric_with_p95[metric_with_p95['is_anomaly']]

agg_cmdb = (
    anomalies_window
    .groupby('cmdb_id', dropna=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        distinct_kpi_count_with_anomalies=('kpi_name', 'nunique')
    )
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
)

# Top 20 cmdb_ids by anomaly_count (compact)
agg_cmdb_top20 = agg_cmdb.head(20)

# 4) For top 5 cmdb_ids, return top 10 KPIs by anomaly point count with earliest anomaly timestamp in window
top5_cmdb_ids = agg_cmdb_top20.head(5)['cmdb_id'].tolist()

kpigroup = (
    anomalies_window[anomalies_window['cmdb_id'].isin(top5_cmdb_ids)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        anomaly_count_for_kpi=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp_in_window=('timestamp_dt', 'min')
    )
    .reset_index()
)

# Format earliest timestamp as string in UTC+8 for compact display
kpigroup['earliest_anomaly_timestamp_in_window'] = kpigroup['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Sort and take top 10 KPIs per cmdb_id
kpigroup_sorted = kpigroup.sort_values(['cmdb_id', 'anomaly_count_for_kpi', 'earliest_anomaly_timestamp_in_window'],
                                       ascending=[True, False, True])
top_kpis_top5 = kpigroup_sorted.groupby('cmdb_id', dropna=False).head(10).reset_index(drop=True)

# Final outputs (compact, JSON-friendly tables)
agg_cmdb_top20, top_kpis_top5

```
Out[2]:
```
```python
# Build a plain-English compact summary string using previously computed variables:
# agg_cmdb_top20, top_kpis_top5
try:
    top20_df = agg_cmdb_top20
    kpis_df = top_kpis_top5
except NameError:
    summary = "Required intermediate variables (agg_cmdb_top20, top_kpis_top5) are not found in the kernel."
else:
    lines = []
    lines.append("Telemetry anomaly summary (metric.csv) — incident window 2022-03-21 15:00:00 to 2022-03-21 15:30:00 CST (UTC+8):")
    lines.append("")
    lines.append("Top 20 components by anomaly count:")
    for i, row in enumerate(top20_df.head(20).itertuples(), start=1):
        lines.append(f"{i}. {row.cmdb_id}: {int(row.anomaly_count)} anomalies, {int(row.distinct_kpi_count_with_anomalies)} distinct KPIs with anomalies")
    lines.append("")
    # Top 5 details
    top5_ids = list(top20_df.head(5)['cmdb_id'])
    lines.append("Top 5 components detail (for each, top KPIs by anomaly point count; format: KPI (count, earliest_anomaly)):")
    for cmdb in top5_ids:
        lines.append(f"- {cmdb}:")
        sub = kpis_df[kpis_df['cmdb_id'] == cmdb].sort_values(['anomaly_count_for_kpi', 'earliest_anomaly_timestamp_in_window'], ascending=[False, True]).head(10)
        if sub.shape[0] == 0:
            lines.append("    (no KPI anomalies found in window)")
            continue
        for r in sub.itertuples(index=False):
            lines.append(f"    - {r.kpi_name} ({int(r.anomaly_count_for_kpi)}, {r.earliest_anomaly_timestamp_in_window})")
    summary = "\n".join(lines)

# Display the compact plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  anomaly_count  distinct_kpi_count_with_anomalies
0                 adservice           8890                                341
4                adservice2           8578                                333
30              frontend2-0           3097                                176
27               frontend-0           2950                                172
28               frontend-1           2930                                172
29               frontend-2           2800                                166
14        checkoutservice-2           2331                                134
12        checkoutservice-0           2325                                134
5              adservice2-0           2229                                102
13        checkoutservice-1           2207                                130
44  productcatalogservice-1           2186                                116
43  productcatalogservice-0           2170                                116
8             cartservice-1           2162                                100
45  productcatalogservice-2           2148                                116
7             cartservice-0           2050                                 96
10           cartservice2-0           1992                                 96
3               adservice-2           1859                                 86
49  recommendationservice-1           1851                                 84
2               adservice-1           1847                                 86
1               adservice-0           1812                                 86,         cmdb_id                                           kpi_name  anomaly_count_for_kpi earliest_anomaly_timestamp_in_window
0     adservice                                        app.grpc.rr                     30              2022-03-21 15:00:00 CST
1     adservice                                     app.http.count                     30              2022-03-21 15:00:00 CST
2     adservice                                        app.http.rr                     30              2022-03-21 15:00:00 CST
3     adservice                                        app.http.sr                     30              2022-03-21 15:00:00 CST
4     adservice    runtime.java_lang_ClassLoading_LoadedClassCount                     30              2022-03-21 15:00:00 CST
..          ...                                                ...                    ...                                  ...
45  frontend2-0  container.node-6.container_fs_io_time_weighted...                     30              2022-03-21 15:00:00 CST
46  frontend2-0   container.node-6.container_fs_limit_MB./dev/vda1                     30              2022-03-21 15:00:00 CST
47  frontend2-0  container.node-6.container_fs_read_seconds./de...                     30              2022-03-21 15:00:00 CST
48  frontend2-0      container.node-6.container_fs_reads./dev/vda1                     30              2022-03-21 15:00:00 CST
49  frontend2-0  container.node-6.container_fs_reads_merged./de...                     30              2022-03-21 15:00:00 CST

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables where possible: metric_with_p95, agg_cmdb_top20, tz
# If they don't exist, load/compute minimally (but they should exist from previous steps).
try:
    metric_with_p95
    agg_cmdb_top20
    tz
except NameError:
    metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    thresholds = (
        metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )
    metric_with_p95 = metric.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    metric_with_p95['timestamp_dt'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
    end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))
    metric_with_p95['is_in_window'] = (metric_with_p95['timestamp_dt'] >= start) & (metric_with_p95['timestamp_dt'] < end)
    metric_with_p95['is_anomaly'] = metric_with_p95['is_in_window'] & (metric_with_p95['value'] >= metric_with_p95['p95'])
    agg_cmdb_top20 = (
        metric_with_p95[metric_with_p95['is_anomaly']]
        .groupby('cmdb_id', dropna=False)
        .agg(anomaly_count=('is_anomaly','sum'), distinct_kpi_count_with_anomalies=('kpi_name','nunique'))
        .reset_index().sort_values('anomaly_count', ascending=False).head(20)
    )

# 1) Select top 10 cmdb_id by anomaly_count
candidate_cmdbs = agg_cmdb_top20.head(10)['cmdb_id'].tolist()

# 2) Detect consecutive anomaly runs within incident window per (cmdb_id, kpi_name)
# Filter to incident window rows only (we'll examine consecutive anomalies there)
start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))

df_win = metric_with_p95[metric_with_p95['is_in_window']].copy()
# Keep only candidate cmdbs to limit scan
df_win = df_win[df_win['cmdb_id'].isin(candidate_cmdbs)].sort_values(['cmdb_id','kpi_name','timestamp_dt'])

# We'll identify consecutive runs of is_anomaly==True within each (cmdb_id,kpi_name).
# Use segment ids via change points.
def extract_runs(g):
    # g is sorted by timestamp_dt
    mask = g['is_anomaly'].fillna(False).astype(bool)
    # If no anomaly at all, return empty DataFrame
    if not mask.any():
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
            'fault_length_minutes','max_value_in_fault','global_P95_threshold'
        ])
    # segment id increments when mask value changes
    seg = (mask != mask.shift(fill_value=False)).cumsum()
    g = g.assign(_seg=seg, _mask=mask)
    # keep only segments where mask True
    segs = g[g['_mask']].groupby('_seg', dropna=False)
    records = []
    for _, segg in segs:
        cmdb = segg['cmdb_id'].iloc[0]
        kpi = segg['kpi_name'].iloc[0]
        start_ts = segg['timestamp_dt'].min()
        end_ts = segg['timestamp_dt'].max()
        length = int(segg.shape[0])  # number of consecutive anomalous points (minutes)
        max_val = float(segg['value'].max())
        p95 = float(segg['p95'].iloc[0])
        records.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'fault_start_timestamp': start_ts,
            'fault_end_timestamp': end_ts,
            'fault_length_minutes': length,
            'max_value_in_fault': max_val,
            'global_P95_threshold': p95
        })
    return pd.DataFrame.from_records(records)

# Apply per (cmdb_id,kpi_name)
runs_list = []
grp = df_win.groupby(['cmdb_id','kpi_name'], sort=False)
for _, g in grp:
    runs = extract_runs(g)
    if not runs.empty:
        runs_list.append(runs)

if runs_list:
    faults_full = pd.concat(runs_list, ignore_index=True)
else:
    faults_full = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'fault_length_minutes','max_value_in_fault','global_P95_threshold'
    ])

# 2b) compute breach_ratio
eps = 1e-9
faults_full['breach_ratio'] = (faults_full['max_value_in_fault'] - faults_full['global_P95_threshold']) / (faults_full['global_P95_threshold'] + eps)

# 3) Aggregate and return top 50 fault records sorted by breach_ratio desc, then fault_length_minutes desc, then earliest fault_start_timestamp
# Ensure sorting keys are appropriate: breach_ratio desc, fault_length_minutes desc, fault_start_timestamp asc
faults_full['fault_start_sort'] = faults_full['fault_start_timestamp']  # datetime tz-aware
faults_sorted = faults_full.sort_values(by=['breach_ratio','fault_length_minutes','fault_start_sort'],
                                        ascending=[False, False, True]).reset_index(drop=True)

# Limit to top 50 and format timestamps as strings for JSON-friendly output
faults_top50 = faults_sorted.head(50).copy()
faults_top50['fault_start_timestamp'] = faults_top50['fault_start_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
faults_top50['fault_end_timestamp'] = faults_top50['fault_end_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order final columns
final_faults = faults_top50[[
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'fault_length_minutes','max_value_in_fault','global_P95_threshold','breach_ratio'
]].reset_index(drop=True)

# Store full intermediate tables in kernel variables (already stored): candidate_cmdbs, faults_full, faults_sorted
# Return only the final faults table (bounded to max 50 rows)
final_faults

```
Out[3]:
```
```python
# Summarize the fault detection results in plain English as a compact string.
# Reuse kernel variables: final_faults, agg_cmdb_top20, candidate_cmdbs, faults_full if available.

try:
    n_candidates = len(candidate_cmdbs)
except NameError:
    candidate_cmdbs = []
    n_candidates = 0

try:
    total_faults_generated = int(faults_full.shape[0])
except NameError:
    total_faults_generated = None

try:
    displayed_faults = final_faults.copy()
    n_displayed = displayed_faults.shape[0]
except NameError:
    displayed_faults = None
    n_displayed = 0

try:
    top_comps = agg_cmdb_top20.head(5).reset_index(drop=True)
except NameError:
    top_comps = None

lines = []
lines.append("Compact summary of detected faults within incident window (based on metric.csv anomalies):")
lines.append("")
lines.append(f"- Candidate components considered: {n_candidates} (top 10 by anomaly count).")
if total_faults_generated is not None:
    lines.append(f"- Total fault runs detected (all candidates, before truncation): {total_faults_generated}")
lines.append(f"- Returning top {n_displayed} fault records (sorted by breach_ratio desc, then length desc).")
lines.append("")
# Top 5 fault highlights
if displayed_faults is not None and n_displayed > 0:
    lines.append("Top 5 faults (cmdb_id, kpi_name, length_min, max_value, P95_threshold, breach_ratio):")
    for r in displayed_faults.head(5).itertuples():
        lines.append(
            f"  - {r.cmdb_id}, {r.kpi_name} — {int(r.fault_length_minutes)} min, "
            f"max={r.max_value_in_fault:.6g}, p95={r.global_P95_threshold:.6g}, breach_ratio={r.breach_ratio:.2e}"
        )
    lines.append("")
    lines.append("Note: Several top faults have global P95 == 0 -> breach_ratio becomes extremely large; treat those as "
                 "cases where historical baseline is zero and any observed value is a breach.")
else:
    lines.append("- No faults detected for the selected candidates in the incident window.")

# Top components by anomaly_count
if top_comps is not None and not top_comps.empty:
    lines.append("")
    lines.append("Top components by anomaly count (from earlier aggregation):")
    for r in top_comps.itertuples():
        lines.append(f"  - {r.cmdb_id}: {int(r.anomaly_count)} anomalies, {int(r.distinct_kpi_count_with_anomalies)} KPIs affected")
else:
    lines.append("- Top components by anomaly count not available.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    fault_start_timestamp      fault_end_timestamp  fault_length_minutes  max_value_in_fault  global_P95_threshold  breach_ratio
0         frontend2-0      container.node-6.container_fs_writes./dev/vda  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                    30            3.000000              0.000000  3.000000e+09
1          adservice2  runtime.java_lang_Memory_ObjectPendingFinaliza...  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                    30            1.583333              0.000000  1.583333e+09
2         frontend2-0   container.node-6.container_fs_writes_MB./dev/vda  2022-03-21 15:00:00 CST  2022-03-21 15:29:00 CST                    30            0.027344              0.000000  2.734375e+07
3   checkoutservice-0  container.node-5.container_network_receive_MB....  2022-03-21 15:03:00 CST  2022-03-21 15:03:00 CST                     1            0.575581              0.058895  8.772958e+00
4   checkoutservice-2  container.node-5.container_network_receive_MB....  2022-03-21 15:21:00 CST  2022-03-21 15:21:00 CST                     1            0.570549              0.059637  8.567038e+00
..                ...                                                ...                      ...                      ...                   ...                 ...                   ...           ...
45        frontend2-0                 container.node-6.container_threads  2022-03-21 15:03:00 CST  2022-03-21 15:04:00 CST                     2           75.000000             52.466667  4.294790e-01
46          adservice                        runtime.jvm_threads_started  2022-03-21 15:08:00 CST  2022-03-21 15:08:00 CST                     1            0.750000              0.525000  4.285714e-01
47         frontend-1   container.node-5.container_memory_working_set_MB  2022-03-21 15:02:00 CST  2022-03-21 15:03:00 CST                     2          104.787109             74.165527  4.128816e-01
48         frontend-1         container.node-5.container_memory_usage_MB  2022-03-21 15:02:00 CST  2022-03-21 15:03:00 CST                     2          104.849609             74.228027  4.125340e-01
49         frontend-2  container.node-5.container_memory_failures.con...  2022-03-21 15:03:00 CST  2022-03-21 15:04:00 CST                     2        25852.500000          18443.050000  4.017475e-01

[50 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse if 'trace' already loaded in kernel; otherwise load from CSV
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) using ALL rows (no time filtering)
trace_thresholds = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# Merge thresholds back to trace (keeps full series)
trace_with_p95 = trace.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Convert timestamps to timezone-aware datetimes in UTC+8 for window filtering and reporting
trace_with_p95['timestamp_dt'] = pd.to_datetime(trace_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window [2022-03-21 15:00:00 CST, 2022-03-21 15:30:00 CST)
start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))

trace_with_p95['is_in_window'] = (trace_with_p95['timestamp_dt'] >= start) & (trace_with_p95['timestamp_dt'] < end)

# 2) Mark anomalies: value >= global_P95 for rows in window
# (per instruction, treat latency/duration/error_rate as >=P95 anomalous; we apply >=P95 universally)
trace_with_p95['is_anomaly'] = trace_with_p95['is_in_window'] & (trace_with_p95['value'] >= trace_with_p95['global_P95'])

# 3) Aggregate anomalies per (cmdb_id, trace_name)
# Consider only rows in window for anomaly_count and max_value_in_window
window_rows = trace_with_p95[trace_with_p95['is_in_window']].copy()

agg = (
    window_rows
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        anomaly_count_in_window=('is_anomaly','sum'),
        earliest_anomaly_timestamp_in_window=('timestamp_dt', lambda s: s[window_rows.loc[s.index,'is_anomaly']].min() if (window_rows.loc[s.index,'is_anomaly'].any()) else pd.NaT),
        global_P95=('global_P95','first'),
        max_value_in_window=('value','max')
    )
    .reset_index()
)

# Keep only trace groups that had at least one anomaly
agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()

# Format earliest timestamp as string for display
agg_anom['earliest_anomaly_timestamp_in_window'] = agg_anom['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 4A) Top 30 trace anomaly rows sorted by earliest_anomaly_timestamp_in_window asc, then anomaly_count_in_window desc
# Convert earliest timestamp back to datetime for sorting where possible (NaT already filtered out)
agg_anom_sorted = agg_anom.sort_values(by=['earliest_anomaly_timestamp_in_window','anomaly_count_in_window'],
                                       ascending=[True, False]).reset_index(drop=True)
final_trace_anom_top30 = agg_anom_sorted.head(30)[[
    'cmdb_id','trace_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','global_P95','max_value_in_window'
]]

# 4B) Distinct cmdb_id list among anomalous traces with their anomaly_count sums, top 20 by anomaly_count
cmdb_agg = (
    agg_anom
    .groupby('cmdb_id', dropna=False)
    .agg(total_anomaly_count=('anomaly_count_in_window','sum'),
         distinct_trace_count_with_anomalies=('trace_name','nunique'))
    .reset_index()
    .sort_values('total_anomaly_count', ascending=False)
)

cmdb_anom_top20 = cmdb_agg.head(20)[['cmdb_id','total_anomaly_count','distinct_trace_count_with_anomalies']]

# Return compact JSON-friendly tables only (store intermediates in kernel variables)
final_trace_anom_top30, cmdb_anom_top20

```
Out[4]:
```
```python
# Summarize the trace anomaly analysis in plain English using existing kernel variables:
# final_trace_anom_top30, cmdb_anom_top20 (created in prior cell).
try:
    top_traces = final_trace_anom_top30
    cmdb_summary = cmdb_anom_top20
except NameError:
    summary = "Required results not found in kernel (final_trace_anom_top30, cmdb_anom_top20)."
else:
    lines = []
    lines.append("Trace anomaly summary (incident window 2022-03-21 15:00:00 to 15:30:00 CST, UTC+8):")
    lines.append("")
    # Overall observation
    lines.append(f"- Found {top_traces.shape[0]} top anomalous trace entries (shown). Many traces show anomalies at the window start (15:00:00).")
    lines.append("- A notable pattern: many trace groups have global_P95 == 0 and max_value_in_window == 0, resulting in anomaly_count = full-window (30 minutes) for error_rate-type traces.")
    lines.append("")
    # Top 5 components by total anomaly count
    lines.append("- Top 5 components by total anomaly counts (component: total_anomaly_count, distinct_trace_count):")
    for r in cmdb_summary.head(5).itertuples(index=False):
        lines.append(f"  - {r.cmdb_id}: {int(r.total_anomaly_count)} anomalies, {int(r.distinct_trace_count_with_anomalies)} distinct traces")
    lines.append("")
    # Earliest anomaly pattern
    earliest = top_traces['earliest_anomaly_timestamp_in_window'].min() if top_traces.shape[0]>0 else None
    if earliest:
        lines.append(f"- Earliest anomalous traces in the top list start at: {earliest}")
    lines.append("")
    lines.append("- Top anomalous trace examples (from the top-30 list):")
    for r in top_traces.head(5).itertuples(index=False):
        lines.append(f"  - {r.cmdb_id} | {r.trace_name} | count={int(r.anomaly_count_in_window)} | earliest={r.earliest_anomaly_timestamp_in_window} | P95={r.global_P95} | max={r.max_value_in_window}")
    lines.append("")
    lines.append("Note: Because many global P95 thresholds are zero, treat zero-baseline trace metrics carefully (could indicate instrumentation/baseline issues).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                  trace_name  anomaly_count_in_window earliest_anomaly_timestamp_in_window  global_P95  max_value_in_window
0     adservice-0            trace.from_frontend-0.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
1     adservice-0            trace.from_frontend-1.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
2     adservice-0            trace.from_frontend-2.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
3     adservice-1            trace.from_frontend-0.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
4     adservice-1            trace.from_frontend-1.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
..            ...                                         ...                      ...                                  ...         ...                  ...
25  cartservice-1     trace.from_checkoutservice-0.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
26  cartservice-1     trace.from_checkoutservice-1.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0
27  cartservice-1  trace.from_checkoutservice-2.duration_mean                       30              2022-03-21 15:00:00 CST         0.0                  0.0
28  cartservice-1   trace.from_checkoutservice-2.duration_p95                       30              2022-03-21 15:00:00 CST         0.0                  0.0
29  cartservice-1     trace.from_checkoutservice-2.error_rate                       30              2022-03-21 15:00:00 CST         0.0                  0.0

[30 rows x 6 columns],                     cmdb_id  total_anomaly_count  distinct_trace_count_with_anomalies
9         checkoutservice-1                 1075                                   92
10        checkoutservice-2                 1036                                   92
8         checkoutservice-0                 1032                                   92
20               frontend-0                  863                                   92
21               frontend-1                  862                                   92
22               frontend-2                  833                                   92
4             cartservice-0                  434                                   28
6             cartservice-2                  422                                   28
30  productcatalogservice-2                  389                                   40
5             cartservice-1                  384                                   28
28  productcatalogservice-0                  380                                   40
29  productcatalogservice-1                  379                                   40
11       checkoutservice2-0                  320                                   36
23              frontend2-0                  299                                   36
34  recommendationservice-2                  272                                   28
32  recommendationservice-0                  270                                   28
33  recommendationservice-1                  265                                   28
12        currencyservice-0                  236                                   24
13        currencyservice-1                  231                                   24
37        shippingservice-1                  230                                   24)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Prefer faults_full (has datetime fields), fall back to final_faults (string timestamps)
try:
    src = faults_full.copy()
except NameError:
    try:
        src = final_faults.copy()
    except NameError:
        src = pd.DataFrame()  # no data available

# Ensure we have the expected columns; if not, return empty table with required schema
cols_needed = ['cmdb_id','kpi_name','fault_start_timestamp','fault_length_minutes',
               'max_value_in_fault','global_P95_threshold','breach_ratio']
if src.empty or not set(cols_needed).intersection(set(src.columns)):
    result_table = pd.DataFrame(columns=['cmdb_id','node_id','kpi_name','fault_start_timestamp',
                                         'fault_length_minutes','max_value_in_fault',
                                         'global_P95_threshold','breach_ratio'])
else:
    # Normalize column names in case faults_full used slightly different names
    df = src.copy()
    # If timestamp is datetime-like, keep as is; if string, parse to tz-aware datetime
    if pd.api.types.is_datetime64_any_dtype(df.get('fault_start_timestamp')):
        dt_series = pd.to_datetime(df['fault_start_timestamp']).dt.tz_convert(tz)
    else:
        # try parsing strings; they may include timezone text like 'CST'
        try:
            dt_series = pd.to_datetime(df['fault_start_timestamp'], utc=True).dt.tz_convert(tz)
        except Exception:
            # fallback: parse naive then localize to tz
            dt_series = pd.to_datetime(df['fault_start_timestamp'], errors='coerce')
            dt_series = dt_series.dt.tz_localize(tz)
    df['fault_start_dt'] = dt_series

    # Filter for frontend pods
    frontend_list = ['frontend-0','frontend-1','frontend-2','frontend2-0']
    df_front = df[df['cmdb_id'].isin(frontend_list)].copy()

    # Extract node id pattern 'node-<number>' from kpi_name
    df_front['node_id'] = df_front['kpi_name'].astype(str).str.extract(r'(node-\d+)', expand=False)
    df_front['node_id'] = df_front['node_id'].where(df_front['node_id'].notna(), None)

    # Ensure numeric columns are correct types
    for numcol in ['fault_length_minutes','max_value_in_fault','global_P95_threshold','breach_ratio']:
        if numcol in df_front.columns:
            df_front[numcol] = pd.to_numeric(df_front[numcol], errors='coerce')

    # Sort by fault_start_timestamp asc then breach_ratio desc
    df_front_sorted = df_front.sort_values(by=['fault_start_dt','breach_ratio'], ascending=[True, False])

    # Format fault_start_timestamp for display as string in UTC+8
    df_front_sorted['fault_start_timestamp'] = df_front_sorted['fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Select and order final columns, limit to max 20 rows
    result_table = df_front_sorted.loc[:, [
        'cmdb_id','node_id','kpi_name','fault_start_timestamp',
        'fault_length_minutes','max_value_in_fault','global_P95_threshold','breach_ratio'
    ]].head(20).reset_index(drop=True)

# Display the compact result table
result_table

```
Out[5]:
```
```python
# Build a compact plain-English summary of the frontend fault records already extracted in `result_table`.
# Reuse kernel variable `result_table`.

try:
    df = result_table.copy()
except NameError:
    summary = "No frontend fault records (result_table not found in kernel)."
else:
    n = df.shape[0]
    cmdbs = df['cmdb_id'].unique().tolist()
    node_ids = sorted({nid for nid in df['node_id'].unique() if nid is not None})
    # Top 5 by breach_ratio for highlight
    top5 = df.sort_values(['fault_start_timestamp','breach_ratio'], ascending=[True, False]).head(5)

    lines = []
    lines.append("Frontend pods fault summary (filtered from detected faults):")
    lines.append(f"- Total matching fault records: {n}")
    lines.append(f"- Pods included: {', '.join(cmdbs)}")
    lines.append(f"- Extracted node_ids from KPIs: {', '.join(node_ids) if node_ids else 'None'}")
    lines.append("")
    lines.append("Top 5 fault highlights (sorted by start time then breach_ratio):")
    for r in top5.itertuples(index=False):
        # shorten kpi_name if long
        kpi = (r.kpi_name if len(r.kpi_name) <= 80 else r.kpi_name[:77] + "...")
        lines.append(
            f"  - {r.cmdb_id} | node={r.node_id} | {kpi} | start={r.fault_start_timestamp} | "
            f"len={int(r.fault_length_minutes)}min | max={r.max_value_in_fault:.6g} | p95={r.global_P95_threshold:.6g} | breach_ratio={r.breach_ratio:.2e}"
        )
    lines.append("")
    lines.append("Observations:")
    lines.append("- Several faults originate on node-6 and node-5 (extracted from KPI names).")
    lines.append("- Some KPIs have global_P95 == 0 which produces extremely large breach_ratio values; treat those as zero-baseline cases.")
    lines.append("- Fault lengths range from short runs (2–4 min) to full-window runs (30 min) for some node-level container KPIs.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id node_id                                           kpi_name    fault_start_timestamp  fault_length_minutes  max_value_in_fault  global_P95_threshold  breach_ratio
0   frontend2-0  node-6      container.node-6.container_fs_writes./dev/vda  2022-03-21 15:00:00 CST                    30        3.000000e+00          0.000000e+00  3.000000e+09
1   frontend2-0  node-6   container.node-6.container_fs_writes_MB./dev/vda  2022-03-21 15:00:00 CST                    30        2.734375e-02          0.000000e+00  2.734375e+07
2    frontend-1  node-5      container.node-5.container_memory_mapped_file  2022-03-21 15:00:00 CST                     4        8.237056e+06          2.891776e+06  1.848442e+00
3    frontend-2    None  mesh.source.frontend.jaeger-collector.istio_re...  2022-03-21 15:00:00 CST                     3        6.252500e-02          5.055000e-02  2.368942e-01
4    frontend-2    None  mesh.source.frontend.currencyservice.istio_req...  2022-03-21 15:00:00 CST                     2        3.239750e-01          2.717187e-01  1.923174e-01
5    frontend-1    None  mesh.destination.unknown.frontend.istio_reques...  2022-03-21 15:00:00 CST                     3        2.526750e+00          2.164025e+00  1.676159e-01
6    frontend-1    None  mesh.source.frontend.jaeger-collector.istio_re...  2022-03-21 15:00:00 CST                     2        8.527500e-02          7.716375e-02  1.051174e-01
7    frontend-1    None  mesh.source.frontend.cartservice.istio_request...  2022-03-21 15:00:00 CST                     2        1.475500e-01          1.344325e-01  9.757685e-02
8    frontend-0    None  mesh.source.frontend.currencyservice.istio_req...  2022-03-21 15:00:00 CST                     2        7.360500e-01          6.707887e-01  9.729032e-02
9    frontend-0  node-5  container.node-5.container_cpu_cfs_throttled_p...  2022-03-21 15:00:00 CST                     2        6.860000e+02          6.378500e+02  7.548797e-02
10   frontend-0  node-5       container.node-5.container_cpu_usage_seconds  2022-03-21 15:00:00 CST                     2        2.743671e+01          2.551293e+01  7.540415e-02
11   frontend-0  node-5         container.node-5.container_cpu_cfs_periods  2022-03-21 15:00:00 CST                     2        6.860000e+02          6.380750e+02  7.510873e-02
12   frontend-0  node-5        container.node-5.container_cpu_user_seconds  2022-03-21 15:00:00 CST                     2        2.736000e+01          2.547900e+01  7.382550e-02
13  frontend2-0    None  mesh.source.frontend2.shippingservice2.istio_r...  2022-03-21 15:00:00 CST                     2        5.522500e-02          5.200750e-02  6.186608e-02
14   frontend-2    None  mesh.source.frontend.cartservice.istio_request...  2022-03-21 15:00:00 CST                     2        9.935000e-02          9.437750e-02  5.268735e-02
15  frontend2-0  node-6                 container.node-6.container_sockets  2022-03-21 15:00:00 CST                    30        2.000000e+01          1.900000e+01  5.263158e-02
16  frontend2-0    None  mesh.source.frontend2.checkoutservice2.istio_r...  2022-03-21 15:00:00 CST                     2        8.500000e+00          8.275000e+00  2.719033e-02
17  frontend2-0    None  mesh.source.frontend2.checkoutservice2.istio_r...  2022-03-21 15:00:00 CST                     2        8.500000e+00          8.275000e+00  2.719033e-02
18  frontend2-0    None  mesh.source.frontend2.checkoutservice2.istio_r...  2022-03-21 15:00:00 CST                     2        8.500000e+00          8.275000e+00  2.719033e-02
19   frontend-1    None  mesh.source.frontend.productcatalogservice.ist...  2022-03-21 15:00:00 CST                     2        1.943875e+00          1.894037e+00  2.631284e-02```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_with_p95 if present; otherwise compute thresholds and anomaly flags
try:
    metric_with_p95
except NameError:
    metric = pd.read_csv('metric.csv')
    # compute global P95 per (cmdb_id, kpi_name)
    thresholds = (
        metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )
    metric_with_p95 = metric.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    metric_with_p95['timestamp_dt'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # incident window
    start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
    end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))
    metric_with_p95['is_in_window'] = (metric_with_p95['timestamp_dt'] >= start) & (metric_with_p95['timestamp_dt'] < end)
    metric_with_p95['is_anomaly'] = metric_with_p95['is_in_window'] & (metric_with_p95['value'] >= metric_with_p95['p95'])

# Ensure timestamp_dt exists and flags exist
if 'timestamp_dt' not in metric_with_p95.columns:
    metric_with_p95['timestamp_dt'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'is_in_window' not in metric_with_p95.columns or 'is_anomaly' not in metric_with_p95.columns:
    start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
    end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))
    metric_with_p95['is_in_window'] = (metric_with_p95['timestamp_dt'] >= start) & (metric_with_p95['timestamp_dt'] < end)
    metric_with_p95['is_anomaly'] = metric_with_p95['is_in_window'] & (metric_with_p95['value'] >= metric_with_p95.get('p95', 0))

# 1) Filter anomaly rows whose kpi_name contains 'node-6' inside the incident window
mask_node6 = metric_with_p95['kpi_name'].astype(str).str.contains('node-6', na=False)
df_node6_anom = metric_with_p95[ metric_with_p95['is_anomaly'] & metric_with_p95['is_in_window'] & mask_node6 ].copy()

# 2) Aggregate by cmdb_id: top 20 by anomaly count
node6_by_cmdb = (
    df_node6_anom
    .groupby('cmdb_id', dropna=False)
    .agg(
        anomaly_count_on_node6_kpis=('is_anomaly', 'sum'),
        distinct_kpi_count_on_node6=('kpi_name', 'nunique')
    )
    .reset_index()
    .sort_values('anomaly_count_on_node6_kpis', ascending=False)
)

node6_by_cmdb_top20 = node6_by_cmdb.head(20).copy()
# Ensure integer types for counts
node6_by_cmdb_top20['anomaly_count_on_node6_kpis'] = node6_by_cmdb_top20['anomaly_count_on_node6_kpis'].astype(int)
node6_by_cmdb_top20['distinct_kpi_count_on_node6'] = node6_by_cmdb_top20['distinct_kpi_count_on_node6'].astype(int)

# 3) Aggregate by kpi_name: top 20 by anomaly count, include earliest timestamp and max value in window
node6_by_kpi = (
    df_node6_anom
    .groupby('kpi_name', dropna=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp_in_window=('timestamp_dt', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
)

# Format earliest timestamp as string for JSON-friendly output
node6_by_kpi['earliest_anomaly_timestamp_in_window'] = node6_by_kpi['earliest_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
node6_by_kpi['anomaly_count'] = node6_by_kpi['anomaly_count'].astype(int)

node6_by_kpi_top20 = node6_by_kpi.head(20)[[
    'kpi_name','anomaly_count','earliest_anomaly_timestamp_in_window','max_value_in_window'
]].reset_index(drop=True)

# Store intermediate full tables in kernel variables (already stored): df_node6_anom, node6_by_cmdb, node6_by_kpi
# Return only the two compact aggregated tables (bounded to 20 rows)
node6_by_cmdb_top20, node6_by_kpi_top20

```
Out[6]:
```
```python
# Summarize node-6 anomaly aggregates (results stored in kernel: node6_by_cmdb_top20, node6_by_kpi_top20)
try:
    cmdb_tbl = node6_by_cmdb_top20
    kpi_tbl = node6_by_kpi_top20
except NameError:
    summary = "Required results (node6_by_cmdb_top20, node6_by_kpi_top20) not found in kernel."
else:
    lines = []
    lines.append("Node-6 KPI anomaly summary (incident window 2022-03-21 15:00:00 to 15:30:00 CST, UTC+8):")
    lines.append("")
    lines.append(f"- Aggregated anomalies on KPIs containing 'node-6' (top {min(20, len(cmdb_tbl))} components):")
    for r in cmdb_tbl.head(10).itertuples(index=False):
        lines.append(f"  - {r.cmdb_id}: {int(r.anomaly_count_on_node6_kpis)} anomalies across {int(r.distinct_kpi_count_on_node6)} distinct node-6 KPIs")
    lines.append("")
    lines.append(f"- Top node-6 KPI names by anomaly count (top {min(20, len(kpi_tbl))} shown):")
    for r in kpi_tbl.head(10).itertuples(index=False):
        lines.append(f"  - {r.kpi_name} | count={int(r.anomaly_count)} | earliest={r.earliest_anomaly_timestamp_in_window} | max={r.max_value_in_window}")
    lines.append("")
    lines.append("Observations:")
    lines.append("- All top node-6 KPIs have anomalies starting at window start (15:00:00) and identical high counts, indicating broad, simultaneous node-6 related anomalies.")
    lines.append("- Several KPI max values are zero while others show very large numbers (e.g., FS limit or CPU quota), suggesting mixed metric types and some zero-baseline instrumentation.")
    lines.append("- Top affected components on node-6 include recommendationservice2-0, cartservice2-0, adservice2-0, productcatalogservice2-0, currencyservice2-0, frontend2-0, and paymentservice2-0.")
    lines.append("")
    lines.append("Recommendation: investigate node-6 (host-level) resource/IO metrics and instrumentation, since many container KPIs referencing 'node-6' are anomalous across multiple pods.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  anomaly_count_on_node6_kpis  distinct_kpi_count_on_node6
6  recommendationservice2-0                         1557                           64
1            cartservice2-0                         1422                           64
0              adservice2-0                         1397                           64
5  productcatalogservice2-0                         1308                           64
2        currencyservice2-0                         1278                           64
3               frontend2-0                         1233                           64
4         paymentservice2-0                         1118                           64,                                              kpi_name  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window
0     container.node-6.container_cpu_load_average_10s            210              2022-03-21 15:00:00 CST             0.000000
1   container.node-6.container_fs_io_time_seconds....            210              2022-03-21 15:00:00 CST             0.000000
2    container.node-6.container_fs_limit_MB./dev/vda1            210              2022-03-21 15:00:00 CST        604630.738281
3   container.node-6.container_fs_read_seconds./de...            210              2022-03-21 15:00:00 CST             0.000000
4   container.node-6.container_fs_io_time_weighted...            210              2022-03-21 15:00:00 CST             0.000000
5   container.node-6.container_fs_inodes_free./dev...            210              2022-03-21 15:00:00 CST             0.000000
6   container.node-6.container_fs_io_current./dev/...            210              2022-03-21 15:00:00 CST             0.000000
7       container.node-6.container_fs_reads./dev/vda1            210              2022-03-21 15:00:00 CST             0.000000
8   container.node-6.container_fs_sector_writes./d...            210              2022-03-21 15:00:00 CST             0.000000
9   container.node-6.container_fs_writes_merged./d...            210              2022-03-21 15:00:00 CST             0.000000
10     container.node-6.container_fs_writes./dev/vda1            210              2022-03-21 15:00:00 CST             0.000000
11  container.node-6.container_fs_write_seconds./d...            210              2022-03-21 15:00:00 CST             0.000000
12  container.node-6.container_fs_reads_merged./de...            210              2022-03-21 15:00:00 CST             0.000000
13  container.node-6.container_fs_sector_reads./de...            210              2022-03-21 15:00:00 CST             0.000000
14             container.node-6.container_threads_max            210              2022-03-21 15:00:00 CST             0.000000
15  container.node-6.container_ulimits_soft.max_op...            210              2022-03-21 15:00:00 CST         65536.000000
16         container.node-6.container_spec_cpu_period            210              2022-03-21 15:00:00 CST        100000.000000
17          container.node-6.container_spec_cpu_quota            210              2022-03-21 15:00:00 CST         30000.000000
18         container.node-6.container_spec_cpu_shares            210              2022-03-21 15:00:00 CST           204.000000
19    container.node-6.container_spec_memory_limit_MB            210              2022-03-21 15:00:00 CST           450.000000)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Incident window
start = tz.localize(pd.Timestamp('2022-03-21 15:00:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 15:30:00'))

# 1) Top 10 cmdb_id with KPI names containing 'node-6' by anomaly count
try:
    # Prefer existing node6_by_cmdb_top20 if present
    node6_by_cmdb_top20
    top10_cmdbs = node6_by_cmdb_top20.head(10)['cmdb_id'].tolist()
except NameError:
    # Derive from metric_with_p95 & anomaly flags
    try:
        metric_with_p95
    except NameError:
        metric_with_p95 = pd.read_csv('metric.csv')
        # If metric_with_p95 lacks flags, compute minimal required fields (but should exist from prior steps)
        thresholds = (
            metric_with_p95.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
            .quantile(0.95).reset_index().rename(columns={'value':'p95'})
        )
        metric_with_p95 = metric_with_p95.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        metric_with_p95['timestamp_dt'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        metric_with_p95['is_in_window'] = (metric_with_p95['timestamp_dt'] >= start) & (metric_with_p95['timestamp_dt'] < end)
        metric_with_p95['is_anomaly'] = metric_with_p95['is_in_window'] & (metric_with_p95['value'] >= metric_with_p95['p95'])
    mask_node6 = metric_with_p95['kpi_name'].astype(str).str.contains('node-6', na=False)
    df_node6_anom_all = metric_with_p95[ metric_with_p95['is_anomaly'] & metric_with_p95['is_in_window'] & mask_node6 ].copy()
    node6_by_cmdb = (
        df_node6_anom_all.groupby('cmdb_id', dropna=False)
        .agg(anomaly_count_on_node6_kpis=('is_anomaly','sum'))
        .reset_index()
        .sort_values('anomaly_count_on_node6_kpis', ascending=False)
    )
    top10_cmdbs = node6_by_cmdb.head(10)['cmdb_id'].tolist()

# Ensure a list; limit length to 10
top10_cmdbs = list(dict.fromkeys(top10_cmdbs))[:10]

# 2) Query error_logs.csv for these cmdb_ids inside incident window
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps in error_logs
error_logs['timestamp_dt'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to window and to top10 cmdbs
err_win = error_logs[
    (error_logs['timestamp_dt'] >= start) & (error_logs['timestamp_dt'] < end) &
    (error_logs['cmdb_id'].isin(top10_cmdbs))
].copy()

# Helper to get up to 3 distinct message previews preserving order
def top_messages(series, limit=3):
    seen = []
    for msg in series:
        if pd.isna(msg):
            continue
        s = str(msg)
        if s not in seen:
            seen.append(s)
        if len(seen) >= limit:
            break
    # truncate each to 140 chars
    previews = [m if len(m) <= 140 else m[:137] + '...' for m in seen]
    # join with separator for compact display
    return " || ".join(previews)

if not err_win.empty:
    err_summary = (
        err_win.groupby('cmdb_id', sort=False)
        .agg(
            error_log_count_in_window=('timestamp_dt','count'),
            earliest_error_log_timestamp_in_window=('timestamp_dt','min'),
            message_previews=('message', lambda s: top_messages(s, limit=3))
        )
        .reset_index()
    )
    err_summary['earliest_error_log_timestamp_in_window'] = err_summary['earliest_error_log_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    # empty structured table
    err_summary = pd.DataFrame(columns=[
        'cmdb_id','error_log_count_in_window','earliest_error_log_timestamp_in_window','message_previews'
    ])

# Limit to top10 and sort by error_log_count desc then earliest timestamp asc
if not err_summary.empty:
    err_summary = err_summary.sort_values(['error_log_count_in_window','earliest_error_log_timestamp_in_window'], ascending=[False,True]).head(10).reset_index(drop=True)
else:
    err_summary = err_summary.head(10)

# 3) Query log.csv rows inside window and aggregate per cmdb_id for top10 cmdbs
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

# Parse timestamps in log.csv
log['timestamp_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

log_win = log[
    (log['timestamp_dt'] >= start) & (log['timestamp_dt'] < end) &
    (log['cmdb_id'].isin(top10_cmdbs))
].copy()

# Ensure value column numeric
log_win['value'] = pd.to_numeric(log_win['value'], errors='coerce').fillna(0)

# Compute sums for log.error_count and log.row_count per cmdb_id
if not log_win.empty:
    log_pivot = (
        log_win.pivot_table(
            index='cmdb_id',
            columns='log_name',
            values='value',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
    )
    # Rename columns to required names; handle missing columns
    log_pivot = log_pivot.rename(columns={
        'log.error_count': 'sum_log_error_count',
        'log.row_count': 'sum_log_row_count'
    })
    if 'sum_log_error_count' not in log_pivot.columns:
        log_pivot['sum_log_error_count'] = 0
    if 'sum_log_row_count' not in log_pivot.columns:
        log_pivot['sum_log_row_count'] = 0

    # earliest_log_timestamp_in_window
    earliest_log_ts = (
        log_win.groupby('cmdb_id', sort=False)['timestamp_dt']
        .min()
        .reset_index()
        .rename(columns={'timestamp_dt':'earliest_log_timestamp_in_window'})
    )
    earliest_log_ts['earliest_log_timestamp_in_window'] = earliest_log_ts['earliest_log_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    log_summary = log_pivot.merge(earliest_log_ts, on='cmdb_id', how='left')
    # Keep only required columns and ensure numeric types
    log_summary = log_summary[['cmdb_id','sum_log_error_count','sum_log_row_count','earliest_log_timestamp_in_window']]
    log_summary['sum_log_error_count'] = log_summary['sum_log_error_count'].astype(int)
    log_summary['sum_log_row_count'] = log_summary['sum_log_row_count'].astype(int)
else:
    log_summary = pd.DataFrame(columns=['cmdb_id','sum_log_error_count','sum_log_row_count','earliest_log_timestamp_in_window'])

# Ensure both tables include all top10 cmdbs (even if zero rows), join to include missing cmdbs with zeros
top10_df = pd.DataFrame({'cmdb_id': top10_cmdbs})

err_summary = top10_df.merge(err_summary, on='cmdb_id', how='left')
err_summary['error_log_count_in_window'] = err_summary['error_log_count_in_window'].fillna(0).astype(int)
err_summary['earliest_error_log_timestamp_in_window'] = err_summary['earliest_error_log_timestamp_in_window'].fillna(pd.NA)
err_summary['message_previews'] = err_summary['message_previews'].fillna('')

log_summary = top10_df.merge(log_summary, on='cmdb_id', how='left')
log_summary['sum_log_error_count'] = log_summary['sum_log_error_count'].fillna(0).astype(int)
log_summary['sum_log_row_count'] = log_summary['sum_log_row_count'].fillna(0).astype(int)
log_summary['earliest_log_timestamp_in_window'] = log_summary['earliest_log_timestamp_in_window'].fillna(pd.NA)

# Keep final outputs compact (max 10 rows each)
error_logs_summary = err_summary.head(10).reset_index(drop=True)
logcsv_summary = log_summary.head(10).reset_index(drop=True)

# Store intermediate tables in kernel variables (df_node6_anom_all, err_win, log_win)
# Return only the two aggregated tables
error_logs_summary, logcsv_summary

```
Out[7]:
```
```python
# Summarize correlation between top node-6 affected pods and logs using kernel tables:
# error_logs_summary, logcsv_summary (created previously).
try:
    err_tbl = error_logs_summary
    log_tbl = logcsv_summary
except NameError:
    summary = "Required tables (error_logs_summary, logcsv_summary) not found in kernel."
else:
    pods = err_tbl['cmdb_id'].tolist()
    total_pods = len(pods)
    # Check if any error logs or log counts exist
    total_error_logs = int(err_tbl['error_log_count_in_window'].sum())
    total_log_errors = int(log_tbl['sum_log_error_count'].sum())
    total_log_rows = int(log_tbl['sum_log_row_count'].sum())

    lines = []
    lines.append("Correlation of top node-6 affected pods with logs (window 2022-03-21 15:00:00 to 15:30:00 CST):")
    lines.append(f"- Top pods considered (up to 10): {', '.join(pods)}")
    lines.append(f"- Number of pods in this list: {total_pods}")
    lines.append("")
    lines.append("Error logs (per pod):")
    lines.append(f"- Total error_log entries across these pods in window: {total_error_logs}")
    lines.append("- Per-pod error_log counts (pod: count):")
    for r in err_tbl.itertuples(index=False):
        lines.append(f"  - {r.cmdb_id}: {int(r.error_log_count_in_window)}")
    lines.append("")
    lines.append("Application logs (per pod):")
    lines.append(f"- Total log.error_count summed: {total_log_errors}")
    lines.append(f"- Total log.row_count summed: {total_log_rows}")
    lines.append("- Per-pod aggregates (pod: sum_log_error_count, sum_log_row_count):")
    for r in log_tbl.itertuples(index=False):
        lines.append(f"  - {r.cmdb_id}: error_count={int(r.sum_log_error_count)}, row_count={int(r.sum_log_row_count)}")
    lines.append("")
    lines.append("Conclusion:")
    if total_error_logs == 0 and total_log_errors == 0 and total_log_rows == 0:
        lines.append("- No error_logs or log.csv entries were found for these top node-6 affected pods within the incident window.")
        lines.append("- Therefore, metrics indicate node-6 related anomalies, but there is no direct evidence in error_logs or aggregated log counts in the same time window.")
        lines.append("- Recommendation: check node-level system logs, node monitoring, and increase log collection for these pods to find root cause.")
    else:
        lines.append("- Some logs were found; see per-pod counts above for details.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  error_log_count_in_window earliest_error_log_timestamp_in_window message_previews
0  recommendationservice2-0                          0                                   <NA>                 
1            cartservice2-0                          0                                   <NA>                 
2              adservice2-0                          0                                   <NA>                 
3  productcatalogservice2-0                          0                                   <NA>                 
4        currencyservice2-0                          0                                   <NA>                 
5               frontend2-0                          0                                   <NA>                 
6         paymentservice2-0                          0                                   <NA>                 ,                     cmdb_id  sum_log_error_count  sum_log_row_count earliest_log_timestamp_in_window
0  recommendationservice2-0                    0                  0                             <NA>
1            cartservice2-0                    0                  0                             <NA>
2              adservice2-0                    0                  0                             <NA>
3  productcatalogservice2-0                    0                  0                             <NA>
4        currencyservice2-0                    0                  0                             <NA>
5               frontend2-0                    0                  0                             <NA>
6         paymentservice2-0                    0                  0                             <NA>)```
```